<a href="https://colab.research.google.com/github/p-tech/wbs-dm-2026/blob/main/fake_data/create_fake_data_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install faker
import random, csv, faker
from datetime import timedelta, datetime
from faker import Faker
from faker.providers import person
from faker.providers import internet
from faker.providers import ssn
from faker.providers import address
from faker.providers import job
from faker.providers import date_time


fake = Faker()
fake.add_provider(person)
fake.add_provider(internet)
fake.add_provider(ssn)
fake.add_provider(address)
fake.add_provider(job)
fake.add_provider(date_time)

def first_name_and_gender():
    g = 'M' if random.randint(0,1) == 0 else 'F'
    n = fake.first_name_male() if g=='M' else fake.first_name_female()
    return {'gender':g,'first_name':n}

def birth_and_start_date():
    sd = fake.date_between(start_date="-20y", end_date="now")
    delta = timedelta(days=365*random.randint(18,40))
    bd = sd-delta

    return {'birth_date':bd.strftime('%m/%d/%Y'), 'start_date': sd.strftime('%m/%d/%Y')}

def birth_and_start_date_on_windows():
    bd = datetime(1960, 1, 1) + timedelta(seconds=random.randint(0,1261600000)) #40 year time delta
    earliest_start_date = bd + timedelta(seconds=random.randint(0,567720000)) #earliest start date is 18 years after birth
    latest_start_date = datetime.now()

    delta = latest_start_date-earliest_start_date
    delta_in_seconds = delta.days*24*60*60+delta.seconds
    random_second = random.randint(0,delta_in_seconds)
    return {'birth_date':bd.strftime('%m/%d/%Y'), 'start_date': (bd+timedelta(seconds=random_second)).strftime('%m/%d/%Y')}

def title_office_org():
    #generate a map of real office to fake office
    offices = ['New York','Austin','Seattle','Chicago','San Francisco']
    #codify the hierarchical structure
    allowed_orgs_per_office = {'New York':['Sales'],'Austin':['Devops','Platform','Product','Internal Tools'],'Chicago':['Devops'], 'Seattle':['Internal Tools','Product'], 'San Francisco':['Management', 'Finance', 'Marketing']}
    allowed_titles_per_org = {
        'Devops':['Engineer','Senior Engineer','Manager'],
        'Sales':['Associate'],
        'Platform':['Engineer'],
        'Product':['Manager','VP'],
        'Internal Tools':['Engineer','Senior Engineer','VP','Manager'],
        'Management':['VP'],
        'Finance':['VP'],
        'Marketing':['VP']
    }

    office = random.choice(offices)
    org = random.choice(allowed_orgs_per_office[office])
    title = random.choice(allowed_titles_per_org[org])
    return {'office':office, 'title':title,'org': org}

def salary_and_bonus():
    salary = round(random.randint(90000,120000)/1000)*1000
    bonus_ratio = random.uniform(0.15,0.2)
    bonus = round(salary*bonus_ratio/500)*500
    return {'salary':salary,'bonus':bonus}

def title_office_org_salary_bonus():
    position = title_office_org()
    title_and_salary_range = {'Engineer':[90,120],'Senior Engineer':[110,140],'Manager':[130,150],'Associate':[60,80],'VP':[150,250]}
    salary_range = title_and_salary_range[position['title']]

    salary = round(random.randint(1000*salary_range[0],1000*salary_range[1])/1000)*1000
    bonus_ratio = random.uniform(0.15,0.2)
    bonus = round(salary*bonus_ratio/500)*500
    position.update({'salary':salary,'bonus':bonus})
    return position

d = dict()
d['first_name_and_gender'] = first_name_and_gender
d['last_name'] = lambda: {'last_name':fake.last_name()}
d['personal_email'] = lambda: {'email':fake.email()}
d['ssn'] = lambda: {'ssn':fake.ssn()}
d['birth_and_start_date'] = birth_and_start_date
d['title_office_org_salary_bonus'] = title_office_org_salary_bonus
d['accrued_holidays'] = lambda: {'accrued_holiday':random.randint(0,20)}

numRows = 100
with open('output.csv', 'w', newline='') as csvfile:
    fieldnames = []
    for k in d.keys():
      fieldnames.extend(list(d[k]().keys()))
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

    writer.writeheader()
    for _ in range(numRows):
        row_dict = {}
        for k in d.keys():
            row_dict.update(d[k]())
        writer.writerow(row_dict)

In [7]:

import sqlite3
import csv

# Connect to the SQLite database (or create it if it doesn't exist)
conn = sqlite3.connect('employee_data.db')
cursor = conn.cursor()

# Create the Employee table (primary key and other employee details)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Employee (
        employee_id INTEGER PRIMARY KEY AUTOINCREMENT,
        first_name TEXT NOT NULL,
        last_name TEXT NOT NULL,
        gender TEXT,
        ssn TEXT UNIQUE,
        birth_date TEXT,
        start_date TEXT,
        personal_email TEXT
    )
''')

# Create the Office table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS Office (
        office_id INTEGER PRIMARY KEY AUTOINCREMENT,
        office_name TEXT UNIQUE NOT NULL
    )
''')

# Create the JobRole table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS JobRole (
        role_id INTEGER PRIMARY KEY AUTOINCREMENT,
        job_title TEXT NOT NULL,
        org TEXT NOT NULL
    )
''')

# Create the EmployeeCompensation table
cursor.execute('''
    CREATE TABLE IF NOT EXISTS EmployeeCompensation (
        compensation_id INTEGER PRIMARY KEY AUTOINCREMENT,
        employee_id INTEGER NOT NULL,
        salary REAL,
        bonus REAL,
        accrued_holidays INTEGER,
        FOREIGN KEY (employee_id) REFERENCES Employee(employee_id)
    )
''')


# Create the EmployeeOffice table (linking table between Employee and Office)
cursor.execute('''
    CREATE TABLE IF NOT EXISTS EmployeeOffice (
        employee_id INTEGER NOT NULL,
        office_id INTEGER NOT NULL,
        FOREIGN KEY (employee_id) REFERENCES Employee(employee_id),
        FOREIGN KEY (office_id) REFERENCES Office(office_id),
        PRIMARY KEY (employee_id, office_id)
    )
''')


# Create the EmployeeRole table (linking table between Employee and JobRole)
cursor.execute('''
  CREATE TABLE IF NOT EXISTS EmployeeRole (
        employee_id INTEGER NOT NULL,
        role_id INTEGER NOT NULL,
        FOREIGN KEY (employee_id) REFERENCES Employee(employee_id),
        FOREIGN KEY (role_id) REFERENCES JobRole(role_id),
        PRIMARY KEY (employee_id, role_id)
    )
''')


# Read data from the CSV file and insert it into the database
with open('output.csv', 'r') as file:
    reader = csv.DictReader(file)
    for row in reader:
      # Insert into Employee table
        cursor.execute('''
            INSERT INTO Employee (first_name, last_name, gender, ssn, birth_date, start_date,personal_email)
            VALUES (?, ?, ?, ?, ?, ?,?)
        ''', (row['first_name'], row['last_name'], row['gender'], row['ssn'], row['birth_date'], row['start_date'], row['email']))

        #Get the employee id
        cursor.execute("SELECT last_insert_rowid()")
        employee_id = cursor.fetchone()[0]


        # Insert or get office ID
        cursor.execute("SELECT office_id FROM Office WHERE office_name = ?", (row['office'],))
        office_data = cursor.fetchone()
        if office_data:
            office_id = office_data[0]
        else:
            cursor.execute("INSERT INTO Office (office_name) VALUES (?)", (row['office'],))
            office_id = cursor.lastrowid

        #Insert into EmployeeOffice
        cursor.execute("INSERT INTO EmployeeOffice (employee_id, office_id) VALUES (?, ?)", (employee_id, office_id))

        # Insert or get role ID
        cursor.execute("SELECT role_id FROM JobRole WHERE job_title = ? AND org = ?", (row['title'], row['org']))
        role_data = cursor.fetchone()
        if role_data:
          role_id = role_data[0]
        else:
          cursor.execute("INSERT INTO JobRole (job_title, org) VALUES (?, ?)", (row['title'], row['org']))
          role_id = cursor.lastrowid

        #Insert into EmployeeRole
        cursor.execute("INSERT INTO EmployeeRole (employee_id, role_id) VALUES (?,?)",(employee_id, role_id))

        #Insert into EmployeeCompensation
        cursor.execute('''
            INSERT INTO EmployeeCompensation (employee_id, salary, bonus, accrued_holidays)
            VALUES (?, ?, ?, ?)
        ''', (employee_id, row['salary'], row['bonus'], row['accrued_holiday']))


# Commit changes and close the connection
conn.commit()
conn.close()


In [ ]:
import sqlite3

conn = sqlite3.connect('employee_data.db')
cursor = conn.cursor()

tables = ['Employee', 'Office', 'JobRole', 'EmployeeCompensation', 'EmployeeOffice', 'EmployeeRole']

for table in tables:
  print(f"First 5 lines of table: {table}")
  cursor.execute(f"SELECT * FROM {table} LIMIT 5")
  results = cursor.fetchall()
  for row in results:
    print(row)
  print("-" * 20)

conn.close()
